In [1]:
import random
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim


#### Neural Net and Replay Buffer

In [2]:
from dqn import DQN, ReplayBuffer

#### Training Loop

In [3]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "code").exists():
    project_root = project_root.parent

if not (project_root / "code").exists():
    raise RuntimeError("Could not locate the project root containing the 'code' directory.")

source_root = str(project_root / "code")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from State.traffic_env import TrafficEnv


env = TrafficEnv()

state_dim = len(env._get_state())
action_dim = len(env.phases)

model = DQN(state_dim, action_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)
buffer = ReplayBuffer()

gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05

episodes = 600

for episode in range(episodes):
    state = env.reset()
    total_reward = 0

    for t in range(env.max_steps):
        
        # epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, action_dim - 1)
        else:
            with torch.no_grad():
                q_values = model(torch.FloatTensor(state))
                action = torch.argmax(q_values).item()

        next_state, reward, done = env.step(action)

        buffer.push((state, action, reward, next_state))
        state = next_state
        total_reward += reward

        if done:
            break

        # training step
        if len(buffer) >= 64:
            states, actions, rewards, next_states = buffer.sample(64)

            states = torch.FloatTensor(states)
            actions = torch.LongTensor(actions)
            rewards = torch.FloatTensor(rewards)
            next_states = torch.FloatTensor(next_states)

            q_values = model(states)
            next_q_values = model(next_states)

            target = rewards + gamma * torch.max(next_q_values, dim=1)[0]

            current_q = q_values.gather(1, actions.unsqueeze(1)).squeeze()

            loss = nn.MSELoss()(current_q, target.detach())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    print(f"Episode {episode}, Reward: {total_reward}")

C:\Users\ahmad\AppData\Local\Temp\ipykernel_10596\2862656382.py:61: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  states = torch.FloatTensor(states)


Episode 0, Reward: -2691.0
Episode 1, Reward: -3562.5
Episode 2, Reward: -2858.5
Episode 3, Reward: -3154.5
Episode 4, Reward: -3751.5
Episode 5, Reward: -2455.5
Episode 6, Reward: -4542.5
Episode 7, Reward: -3263.5
Episode 8, Reward: -4683.5
Episode 9, Reward: -3250.5
Episode 10, Reward: -2967.0
Episode 11, Reward: -3546.0
Episode 12, Reward: -3869.5
Episode 13, Reward: -4182.0
Episode 14, Reward: -4109.0
Episode 15, Reward: -3065.0
Episode 16, Reward: -2443.0
Episode 17, Reward: -3258.5
Episode 18, Reward: -2745.5
Episode 19, Reward: -3113.5
Episode 20, Reward: -4271.0
Episode 21, Reward: -3030.5
Episode 22, Reward: -3351.5
Episode 23, Reward: -4050.0
Episode 24, Reward: -2569.5
Episode 25, Reward: -2460.0
Episode 26, Reward: -3460.5
Episode 27, Reward: -2604.0
Episode 28, Reward: -3743.0
Episode 29, Reward: -2939.0
Episode 30, Reward: -3064.0
Episode 31, Reward: -3671.5
Episode 32, Reward: -3417.5
Episode 33, Reward: -3066.5
Episode 34, Reward: -2233.0
Episode 35, Reward: -1986.5
Ep

#### Save Model

In [4]:
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "code").exists():
    project_root = project_root.parent

if not (project_root / "code").exists():
    raise RuntimeError("Could not locate the project root containing the 'code' directory.")

model_path = project_root / "code" / "Neural_Networks" / "DQN_Implementation" / "traffic_dqn_model.pth"
torch.save(model.state_dict(), model_path)
print(f"Saved model to {model_path}")

Saved model to C:\Users\ahmad\OneDrive\Desktop\Direct\Sping2026\CPE_FinalProject\Neural-Network-Application-in-Traffic-Management-CPE-593-WS-\code\Neural_Networks\DQN_Implementation\traffic_dqn_model.pth
